In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

import glob
import os
import csv
import ast

from scipy.stats import chi2_contingency
from math import sqrt

from itertools import combinations

### Functions - Global 

In [ ]:
def assign_season(month):
    if month in [12, 1, 2, 3]:
        return 'Dry_Season'
    elif month in [4, 5, 6, 7]:
        return 'Major_Rainy_Season'
    elif month in [9, 10, 11]:
        return 'Minor_Rainy_Season'
    else:
        return 'Transition_Season'

### Reading Temp & Precip (using 6km buffer version)

In [ ]:
temp_all = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/temp_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
temp_all['time'] = pd.to_datetime(temp_all['time'])

In [ ]:
precip_all = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/precip_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
precip_all['time'] = pd.to_datetime(precip_all['time'])

### Wind (using 6km buffer version)

In [ ]:
hourly_wind_df = pd.read_csv('/home/kdonkor_umass_edu/Interpolation/ea_hourly_wind_avg_6km.csv').drop(columns=['Unnamed: 0'])

hourly_wind_df['Timestamp'] = pd.to_datetime(hourly_wind_df['Timestamp'])
hourly_wind_df = hourly_wind_df.rename(columns = {'Timestamp':'time'})
hourly_wind_df['Date'] = hourly_wind_df['time'].dt.date

### Lightning (using version aligned with 6km buffer)

In [ ]:
hourly_lightning_df = pd.read_csv('/home/kdonkor_umass_edu/Interpolation/ea_hourly_lightning_avg_6km_buffer_aligned.csv').drop(columns=['Unnamed: 0'])

hourly_lightning_df['Timestamp'] = pd.to_datetime(hourly_lightning_df['Timestamp'])
hourly_lightning_df = hourly_lightning_df.rename(columns = {'Timestamp':'time'})
hourly_lightning_df['Date'] = hourly_lightning_df['time'].dt.date

### Extreme Weather Definitions 

### Temp 

In [ ]:
temp_all['Date'] = temp_all['time'].dt.floor('D')

# Hot hour flag
temp_all['Hot_Hour'] = temp_all['Temp'] > 32

# Group by EA and Date, and count Hot_Hour sum
daily_hot_hours = (
    temp_all.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Hot_Hour': 'sum'})
)

daily_hot_hours = daily_hot_hours.rename(columns={'Hot_Hour': 'Num_Hot_Hours'})

### Precip 

In [ ]:
precip_all['Date'] = precip_all['time'].dt.floor('D')  

# Group by EA and Date, sum Precip
daily_precip = (
    precip_all.groupby(['ea_code9ch', 'Date'], as_index=False)
      .agg({'Precip': 'sum'})
)

### Wind 

In [ ]:
# Windy hour flag
hourly_wind_df['Windy_Hour'] = hourly_wind_df['Wind Gusts (m/s)'] > 5.93

# Group by EA and Date, and count Windy_Hour sum
daily_windy_hours = (
    hourly_wind_df.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Windy_Hour': 'sum'})
)

daily_windy_hours = daily_windy_hours.rename(columns={'Windy_Hour': 'Num_Windy_Hours'})

### Lightning 

In [ ]:
daily_lightning_per_ea = hourly_lightning_df.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()

### Extreme Weather Percentiles 

In [ ]:
### 90th percentile 

hot_hrs_90_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.90)
temp_90_thresh = temp_all['Temp'].quantile(0.90)

precip_90_thresh = daily_precip['Precip'].quantile(0.90)
lightning_90_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.90)

windy_hrs_90_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.90)
wind_90_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.90)

# --- # 

### 95th percentile 

hot_hrs_95_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.95)
temp_95_thresh = temp_all['Temp'].quantile(0.95)

precip_95_thresh = daily_precip['Precip'].quantile(0.95)
lightning_95_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.95)

windy_hrs_95_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.95)
wind_95_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.95)

# --- # 

### 99th percentile 

hot_hrs_99_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.99)
temp_99_thresh = temp_all['Temp'].quantile(0.99)

precip_99_thresh = daily_precip['Precip'].quantile(0.99)
lightning_99_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.99)

windy_hrs_99_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.99)
wind_99_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.99)

#### Dictionary for percentiles 

In [ ]:
# Percentile Threshold dictionaries 
temp_thresh_dict = {
    '90': temp_90_thresh,
    '95': temp_95_thresh,
    '99': temp_99_thresh
}

hot_hrs_thresh_dict = {
    '90': hot_hrs_90_thresh,
    '95': hot_hrs_95_thresh,
    '99': hot_hrs_99_thresh
}

precip_thresh_dict = {
    '90': precip_90_thresh,
    '95': precip_95_thresh,
    '99': precip_99_thresh
}

wind_thresh_dict = {
    '90': wind_90_thresh,
    '95': wind_95_thresh,
    '99': wind_99_thresh
}

windy_hrs_thresh_dict = {
    '90': windy_hrs_90_thresh,
    '95': windy_hrs_95_thresh,
    '99': windy_hrs_99_thresh
}

lightning_thresh_dict = {
    '90': 1,   # at least 1 lightning strike 
    '95': lightning_95_thresh,
    '99': lightning_99_thresh
}

## --- Co_occurrence per EA Workflow --- 

### EAs n Sites Mapping (using EAs within 6km buffer) 

In [ ]:
merged_eas_sites = pd.read_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/ea_site_list_6km_buffer.csv')
merged_eas_sites = merged_eas_sites[['ea_code9ch', 'Intersecting_Sites']]

# Convert the string representation of lists to actual lists
merged_eas_sites['Intersecting_Sites'] = merged_eas_sites['Intersecting_Sites'].apply(ast.literal_eval)

## Outage Workflow 

### PQR hourly data  

In [ ]:
## 22 
pqr_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2022/merged_outage_n_voltage_hourly_22_NEW.csv')
pqr_hourly_22['time'] = pd.to_datetime(pqr_hourly_22['time'])
pqr_hourly_22['time'] = pqr_hourly_22['time'].dt.tz_convert(None)

## 23 
pqr_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2023/merged_outage_n_voltage_hourly_23_NEW.csv')
pqr_hourly_23['time'] = pd.to_datetime(pqr_hourly_23['time'])
pqr_hourly_23['time'] = pqr_hourly_23['time'].dt.tz_convert(None)

pqr_hourly_all = pd.concat([pqr_hourly_22, pqr_hourly_23], ignore_index=True).drop(columns=['Unnamed: 0'])

### Remove sites with less than TWO YEARS worth of data 

In [ ]:
site_start_dates = pqr_hourly_all.groupby('site_id')['time'].min()

# Filter for sites that start in Jan 2022
sites_with_full_data = site_start_dates[site_start_dates.dt.to_period('M') == '2022-01'].index

monthly_sums = (
    pqr_hourly_all
    .groupby(['site_id', pd.Grouper(key='time', freq='ME')])  # replace with actual name
    .sum()
    .reset_index()
)

# remove site '0' 
monthly_sums = monthly_sums[~(monthly_sums['site_id'] == 0)].reset_index(drop = True)

In [ ]:
def get_incomplete_sites(df, expected_months=24):
    incomplete_sites = []
    for site in df['site_id'].unique():
        count = df[df['site_id'] == site].shape[0]
        if count < expected_months:
            print(f"Site ID {site}: {count} months (incomplete)")
            incomplete_sites.append(site)
    return incomplete_sites

In [ ]:
# Sites with less than 24 months 
incomplete_site_ids = get_incomplete_sites(monthly_sums)

pqr_hourly_all = pqr_hourly_all[ ~(pqr_hourly_all['site_id'].isin(incomplete_site_ids)) ]

### Functions 

In [ ]:
def prepare_hourly_all_weather_outage_data(ea_row, temp_df, precip_df, lightning_df, wind_df, pqr_df, outage_threshold):
    ea = ea_row['ea_code9ch']
    site_list = ea_row['Intersecting_Sites']
    
    # Loop through each site
    for site_id in site_list:
        site_id = int(site_id)  # Ensure site IDs are integers

        # Filter temperature data
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        temp_filt['time'] = temp_filt['time'].astype('datetime64[ns]')

        # Filter precipitation data
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        precip_filt['time'] = precip_filt['time'].astype('datetime64[ns]')
        precip_filt = precip_filt[['time', 'Precip']]

        
        # Filter lightning data
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        lightning_filt = lightning_filt[['time', 'Lightning Events']]  

        # Filter wind data
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        wind_filt = wind_filt[['time', 'Wind Gusts (m/s)']]  

        
        # Filter outage data
        pqr_filt = (
            pqr_df[pqr_df['site_id'] == site_id]
            .reset_index(drop=True)[['time', 'site_id', 'outage_events', 'outage_mins']]
        )

        # Flag outages
        flagged_outages = flag_outage_hours(pqr_filt, threshold=outage_threshold)

        # Merge all
        merged = (
            temp_filt
            .merge(precip_filt, on='time', how='inner')
            .merge(lightning_filt, on='time', how='inner')
            .merge(wind_filt, on='time', how='inner')
            .merge(flagged_outages, on='time', how='inner')
        )
        merged['ea_code9ch'] = ea
        merged['site_id'] = site_id  # Add site_id to the merged data

        return merged

In [ ]:
## Specify Outage Duration (to be classified as outage or not) 

def flag_outage_hours(df, threshold):
    
    df = df.copy()
    df['Outage_Flag'] = df['outage_mins'] >= threshold
    df['Outage_Dur'] = round( (threshold/60), 2)
    df = df[['time', 'site_id', 'Outage_Flag', 'Outage_Dur']]
    
    return df

In [ ]:
def create_daily_summary_all_weather(
    df, 
    temp_thresh=32, 
    hot_hours_thresh=5, 
    precip_thresh=15, 
    lightning_thresh=42, 
    wind_thresh=5.93, 
    windy_hours_thresh=6
):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh
    df['Windy_Hour'] = df['Wind Gusts (m/s)'] > wind_thresh

    # Aggregate at EA-day level (not site-day)
    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Outage_Flag': lambda x: (x > 0).any(),   # EA outage if any site outage
              'Outage_Dur': 'mean',                    
              'Precip': 'sum',               
              'Lightning Events': 'sum',        
              'Hot_hour': 'sum',                
              'Windy_Hour': 'sum'                 
          })
    )

    # Classify EA-days as hazard days
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Windy_Day'] = daily_summary['Windy_Hour'] >= windy_hours_thresh
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Add Season Column
    daily_summary['Month'] = pd.to_datetime(daily_summary['Date']).dt.month
    daily_summary['Season'] = daily_summary['Month'].apply(assign_season)

    return daily_summary

In [ ]:
def calculate_cooccurrence_prob_all_combinations_per_EA(
    df, 
    print_top_n=10, 
    filter_top_n_only=True, 
    print_names=True,
    ea_col='ea_code9ch'
):
    df = df.copy()

    # Normalize season labels
    df['Season'] = df['Season'].replace({
        'Major_Rainy_Season': 'Rainy_Season',
        'Minor_Rainy_Season': 'Rainy_Season'
    })

    df['Outage_Day'] = df['Outage_Flag'] > 0

    # Define hazards and their columns
    hazard_cols = {
        'T': 'Hot_Day',
        'P': 'Rainy_Day',
        'W': 'Windy_Day',
        'L': 'Extreme_Lightning_Day'
    }
    hazard_keys = list(hazard_cols.keys())

    # Generate all non-empty combinations of hazards
    all_combos = []
    for r in range(1, len(hazard_keys) + 1):
        all_combos.extend(combinations(hazard_keys, r))

    def p_cond(n, d): 
        return n / d if d else np.nan

    all_results = []

    # Seasons to compute (including "All_Seasons")
    seasons_to_compute = df['Season'].unique().tolist() + ["All_Seasons"]

    # Loop over each EA
    for ea in df[ea_col].unique():
        ea_df_all = df[df[ea_col] == ea].copy()

        for season in seasons_to_compute:
            if season == "All_Seasons":
                ea_df = ea_df_all.copy()
            else:
                ea_df = ea_df_all[ea_df_all['Season'] == season].copy()

            if ea_df.empty:
                continue

            # Track combo counts for filtering
            combo_counts = {}

            # Create exclusive hazard columns for this EA+Season
            for combo in all_combos:
                combo_name = ''.join(combo)
                in_combo = np.logical_and.reduce([ea_df[hazard_cols[h]] for h in combo])
                not_in_combo = np.logical_not(np.logical_or.reduce([ea_df[hazard_cols[h]] for h in hazard_keys if h not in combo]))
                ea_df[f'{combo_name}_Only_Day'] = in_combo & not_in_combo
                ea_df[f'{combo_name}_Only_Outage'] = ea_df[f'{combo_name}_Only_Day'] & ea_df['Outage_Day']
                combo_counts[combo_name] = ea_df[f'{combo_name}_Only_Day'].sum()

            # Identify top N for this EA+Season
            top_combos = sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:print_top_n]
            top_combo_names = {name for name, _ in top_combos}

            # Calculate summary for this EA+Season
            ea_row = {
                'ea_code9ch': ea,
                'season': season,
                'total_days': len(ea_df),
                'outage_dur': ea_df['Outage_Dur'].mean()
            }

            for combo in all_combos:
                combo_name = ''.join(combo)
                if filter_top_n_only and combo_name not in top_combo_names:
                    continue

                days_sum = ea_df[f'{combo_name}_Only_Day'].sum()
                outage_sum = ea_df[f'{combo_name}_Only_Outage'].sum()

                ea_row[f'{combo_name}_days'] = days_sum
                ea_row[f'{combo_name}_n_outage'] = outage_sum
                ea_row[f'{combo_name}_outage_prob'] = p_cond(outage_sum, days_sum)

            all_results.append(ea_row)

    return pd.DataFrame(all_results).fillna(0).round(4)

In [ ]:
filtered_eas_sites_copy_r1 = merged_eas_sites.copy()

### *** Using the TPLW dataset *** 

### 1+ hour outages 

#### 95th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 60  # duration in minutes
percentile = '95'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_all_weather_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [ ]:
daily_agg_global = create_daily_summary_all_weather(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
    wind_thresh=wind_thresh_dict[percentile], 
    windy_hours_thresh=windy_hrs_thresh_dict[percentile]
)

rez_co_occurrence_1hr_global_95th_per_EA = calculate_cooccurrence_prob_all_combinations_per_EA(daily_agg_global)
rez_co_occurrence_1hr_global_95th_per_EA['Pct'] = f"{percentile}th"

rez_co_occurrence_1hr_global_95th_per_EA = rez_co_occurrence_1hr_global_95th_per_EA[
                                                rez_co_occurrence_1hr_global_95th_per_EA
                                                    ['season'] == 'All_Seasons'].reset_index(drop=True)


### Filter columns of interest 
outage_cols_of_interest_TPLW = ['ea_code9ch', 'outage_dur', 'total_days', 
                           'PWL_days', 'PWL_n_outage', 'PWL_outage_prob', 
                           'Pct']

outage_ea_co_occur_95 = rez_co_occurrence_1hr_global_95th_per_EA[outage_cols_of_interest_TPLW]

### Merge SPATIAL Co_occurrence Dfs 

In [ ]:
rez_SPATIAL_1hr_list = [outage_ea_co_occur_95]
rez_spatial_1hr_TPLW = pd.concat(rez_SPATIAL_1hr_list, ignore_index=True).fillna(0)

# rez_spatial_1hr_TPLW.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_spatial_1hr_TPLW.csv')

### *** Using the TPL dataset *** 

In [ ]:
def prepare_hourly_df_TPL_n_outage_data(ea_row, temp_df, precip_df, lightning_df, pqr_df, outage_threshold):
    ea = ea_row['ea_code9ch']
    site_list = ea_row['Intersecting_Sites']

    all_merged = []

    for site_id in site_list:
        site_id = int(site_id)

        # Filter temperature
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])

        temp_filt['time'] = temp_filt['time'].astype('datetime64[ns]')

        # Filter precipitation
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        precip_filt['time'] = precip_filt['time'].astype('datetime64[ns]')
        precip_filt = precip_filt[['time', 'Precip']]

        # Merge temp and precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # Filter lightning and outer join
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        # Filter outage data and flag
        pqr_filt = (
            pqr_df[pqr_df['site_id'] == site_id]
            .reset_index(drop=True)[['time', 'site_id', 'outage_events', 'outage_mins']]
        )
        flagged_outages = flag_outage_hours(pqr_filt, threshold=outage_threshold)

        # Inner join outages
        merged = merged.merge(flagged_outages, on='time', how='inner')

        # Add identifiers
        merged['ea_code9ch'] = ea
        merged['site_id'] = site_id

        # Reorder columns
        merged = merged[['time', 'Date', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
                         'Lightning Events', 'Outage_Flag', 'Outage_Dur']]

        # merged = merged[['time', 'Date', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
        #          'Lightning Events']]

        all_merged.append(merged)

    return pd.concat(all_merged, ignore_index=True)

In [ ]:
def create_daily_summary_TPL(df, temp_thresh, hot_hours_thresh, precip_thresh, lightning_thresh):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh

    # Aggregate at EA-day level instead of site-day
    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Outage_Flag': lambda x: (x > 0).any(),   # outage if any site has outage
              'Outage_Dur': 'mean',           
              'Precip': 'sum',                          
              'Lightning Events':'sum',              
              'Hot_hour': 'sum',                       
          })
    )

    # Classify Days as High Temperature, High Precipitation or Extreme Lightning
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Add Season Column
    daily_summary['Month'] = pd.to_datetime(daily_summary['Date']).dt.month
    daily_summary['Season'] = daily_summary['Month'].apply(assign_season)

    return daily_summary

In [ ]:
def calculate_cooccurrence_prob_all_combinations_TPL(
    df, 
    print_top_n=10, 
    filter_top_n_only=True, 
    print_names=False,
    ea_col='ea_code9ch'
):
    df = df.copy()

    # Normalize season labels (merge Major/Minor rainy, keep Transition)
    df['Season'] = df['Season'].replace({
        'Major_Rainy_Season': 'Rainy_Season',
        'Minor_Rainy_Season': 'Rainy_Season'
    })

    df['Outage_Day'] = df['Outage_Flag'] > 0

    # Define hazards and their columns (TPL only)
    hazard_cols = {
        'T': 'Hot_Day',
        'P': 'Rainy_Day',
        'L': 'Extreme_Lightning_Day'
    }
    hazard_keys = list(hazard_cols.keys())

    # Generate all non-empty combinations of hazards
    all_combos = []
    for r in range(1, len(hazard_keys) + 1):
        all_combos.extend(combinations(hazard_keys, r))

    def p_cond(n, d): 
        return n / d if d else np.nan

    all_results = []

    # Seasons to compute (unique + All_Seasons)
    seasons_to_compute = df['Season'].unique().tolist() + ["All_Seasons"]

    # Loop over each EA
    for ea in df[ea_col].unique():
        ea_df_all = df[df[ea_col] == ea].copy()

        for season in seasons_to_compute:
            if season == "All_Seasons":
                ea_df = ea_df_all.copy()
            else:
                ea_df = ea_df_all[ea_df_all['Season'] == season].copy()

            if ea_df.empty:
                continue

            # Track combo counts for filtering
            combo_counts = {}

            # Create exclusive hazard columns for this EA + Season
            for combo in all_combos:
                combo_name = ''.join(combo)
                in_combo = np.logical_and.reduce([ea_df[hazard_cols[h]] for h in combo])
                not_in_combo = np.logical_not(np.logical_or.reduce([ea_df[hazard_cols[h]] for h in hazard_keys if h not in combo]))
                ea_df[f'{combo_name}_Only_Day'] = in_combo & not_in_combo
                ea_df[f'{combo_name}_Only_Outage'] = ea_df[f'{combo_name}_Only_Day'] & ea_df['Outage_Day']
                combo_counts[combo_name] = ea_df[f'{combo_name}_Only_Day'].sum()

            # Identify top N for this EA + Season
            top_combos = sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:print_top_n]
            top_combo_names = {name for name, _ in top_combos}

            # Calculate summary for this EA + Season
            ea_row = {
                ea_col: ea,
                'season': season,
                'total_days': len(ea_df),
                'outage_dur': ea_df['Outage_Dur'].mean()
            }

            for combo in all_combos:
                combo_name = ''.join(combo)
                if filter_top_n_only and combo_name not in top_combo_names:
                    continue

                days_sum = ea_df[f'{combo_name}_Only_Day'].sum()
                outage_sum = ea_df[f'{combo_name}_Only_Outage'].sum()

                ea_row[f'{combo_name}_days'] = days_sum
                ea_row[f'{combo_name}_n_outage'] = outage_sum
                ea_row[f'{combo_name}_outage_prob'] = p_cond(outage_sum, days_sum)

            all_results.append(ea_row)

    return pd.DataFrame(all_results).fillna(0).round(4)

#### 95th percentile 

In [ ]:
## Outage Duration & Percentile 
dur = 60  # duration in minutes
percentile = '95'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPL_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [ ]:
daily_agg_global = create_daily_summary_TPL(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
)

rez_co_occurrence_1hr_global_95th_per_EA = calculate_cooccurrence_prob_all_combinations_TPL(daily_agg_global)
rez_co_occurrence_1hr_global_95th_per_EA['Pct'] = f"{percentile}th"

rez_co_occurrence_1hr_global_95th_per_EA = rez_co_occurrence_1hr_global_95th_per_EA[
                                                rez_co_occurrence_1hr_global_95th_per_EA
                                                    ['season'] == 'All_Seasons'].reset_index(drop=True)


### Filter columns of interests 
outage_cols_of_interest_TPL = ['ea_code9ch', 'outage_dur', 'total_days', 
                          'T_days', 'T_n_outage', 'T_outage_prob', 
                          'P_days', 'P_n_outage', 'P_outage_prob', 
                           'L_days', 'L_n_outage', 'L_outage_prob', 
                           'TL_days', 'TL_n_outage', 'TL_outage_prob', 
                           'PL_days', 'PL_n_outage', 'PL_outage_prob', 
                           'Pct']


outage_ea_co_occur_95 = rez_co_occurrence_1hr_global_95th_per_EA[outage_cols_of_interest_TPL]

### Merge SPATIAL Co_occurrence Dfs - TPL 

In [ ]:
rez_SPATIAL_1hr_list = [outage_ea_co_occur_95]
rez_spatial_1hr_TPL = pd.concat(rez_SPATIAL_1hr_list, ignore_index=True).fillna(0)

# rez_spatial_1hr_TPL.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_spatial_1hr_TPL.csv')

### 8+ hour outages  

In [ ]:
## Outage Duration & Percentile 
dur = 480  # duration in minutes
percentile = '95'

# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPL_n_outage_data(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        pqr_hourly_all, 
        outage_threshold=dur
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [ ]:
daily_agg_global = create_daily_summary_TPL(
    merged_hourly_data_global, 
    temp_thresh=temp_thresh_dict[percentile], 
    hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
    precip_thresh=precip_thresh_dict[percentile], 
    lightning_thresh=lightning_thresh_dict[percentile],  
)

rez_co_occurrence_8hr_global_95th_per_EA = calculate_cooccurrence_prob_all_combinations_TPL(daily_agg_global)

rez_co_occurrence_8hr_global_95th_per_EA['Pct'] = f"{percentile}th"

rez_co_occurrence_8hr_global_95th_per_EA = rez_co_occurrence_8hr_global_95th_per_EA[
                                                rez_co_occurrence_8hr_global_95th_per_EA
                                                    ['season'] == 'All_Seasons'].reset_index(drop=True)


### Filter columns of interest 
outage_ea_co_occur_95_8hr = rez_co_occurrence_8hr_global_95th_per_EA[outage_cols_of_interest_TPL]

### Merge SPATIAL Co_occurrence Dfs - TPL 

In [ ]:
rez_SPATIAL_8hr_list = [outage_ea_co_occur_95_8hr]
rez_spatial_8hr_TPL = pd.concat(rez_SPATIAL_8hr_list, ignore_index=True).fillna(0)

# rez_spatial_8hr_TPL.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_spatial_8hr_TPL.csv')

## Undervoltages Workflow 

In [ ]:
def flag_undervolt_hours(df):
    df = df.copy()
    df['Undervolt_Flag'] = df['total_undervolt_events'] > 0
    df['Undervolt_Dur'] = round(df['total_undervolt_duration'] / 60, 2)
    df = df[['time', 'site_id', 'Undervolt_Flag', 'Undervolt_Dur']]
    return df

In [ ]:
def prepare_hourly_df_TPL_n_undervolt_data(ea_row, temp_df, precip_df, lightning_df, pqr_df):
    ea = ea_row['ea_code9ch']
    site_list = ea_row['Intersecting_Sites']

    all_merged = []

    for site_id in site_list:
        site_id = int(site_id)

        # Filter temperature
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])

        temp_filt['time'] = temp_filt['time'].astype('datetime64[ns]')

        # Filter precipitation
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
        )
        precip_filt['time'] = precip_filt['time'].astype('datetime64[ns]')
        precip_filt = precip_filt[['time', 'Precip']]

        # Merge temp and precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # Filter lightning and outer join
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .reset_index(drop=True)
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        # Filter outage data and flag
        pqr_filt = (
            pqr_df[pqr_df['site_id'] == site_id]
            .reset_index(drop=True)[['time', 'site_id', 'total_undervolt_events', 'total_undervolt_duration']]
        )

         # Flag undervolts
        flagged_undervolts = flag_undervolt_hours(pqr_filt)
        
        # Inner join outages
        merged = merged.merge(flagged_undervolts, on='time', how='inner')
        
        # Add identifiers
        merged['ea_code9ch'] = ea
        merged['site_id'] = site_id

        # Reorder columns
        merged = merged[['time', 'Date', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
                         'Lightning Events', 'Undervolt_Flag', 'Undervolt_Dur']]

        all_merged.append(merged)

    return pd.concat(all_merged, ignore_index=True)

In [ ]:
def create_daily_summary_unv_TPL(df, temp_thresh, hot_hours_thresh, precip_thresh, lightning_thresh):
    df = df.copy()
    df['Date'] = df['time'].dt.date
    df['Hot_hour'] = df['Temp'] > temp_thresh

    
    daily_summary = (
        df.groupby(['Date', 'ea_code9ch'], as_index=False)
          .agg({
              'Undervolt_Flag': lambda x: (x > 0).any(),  # True if any site had undervolt
              'Undervolt_Dur': 'sum',                   
              'Precip': 'sum',
              'Lightning Events': 'sum',
              'Hot_hour': 'sum',
          })
    )

    # Classify Days as High Temperature, High Precipitation or Extreme Lightning
    daily_summary['Hot_Day'] = daily_summary['Hot_hour'] >= hot_hours_thresh
    daily_summary['Rainy_Day'] = daily_summary['Precip'] >= precip_thresh
    daily_summary['Extreme_Lightning_Day'] = daily_summary['Lightning Events'] >= lightning_thresh

    # Add Season Column
    daily_summary['Month'] = pd.to_datetime(daily_summary['Date']).dt.month
    daily_summary['Season'] = daily_summary['Month'].apply(assign_season)

    return daily_summary

In [ ]:
#### Calculate co-occurrence PER EA 

def calculate_cooccurrence_prob_all_combinations_undervolt_per_EA(
    df, 
    undervolt_dur=20,              
    print_top_n=10, 
    filter_top_n_only=True, 
    print_names=False,
    ea_col='ea_code9ch'
):
    df = df.copy()

    # Normalize season labels (merge Major/Minor rainy, keep Transition)
    df['Season'] = df['Season'].replace({
        'Major_Rainy_Season': 'Rainy_Season',
        'Minor_Rainy_Season': 'Rainy_Season'
    })

    # Define undervolt day flag
    df['Undervolt_Day'] = df['Undervolt_Flag'] > 0  

    # Define hazards and their columns
    hazard_cols = {
        'T': 'Hot_Day',
        'P': 'Rainy_Day',
        'L': 'Extreme_Lightning_Day'
    }
    hazard_keys = list(hazard_cols.keys())

    # Generate all non-empty combinations of hazards
    all_combos = []
    for r in range(1, len(hazard_keys) + 1):
        all_combos.extend(combinations(hazard_keys, r))

    def p_cond(n, d): 
        return n / d if d else np.nan

    all_results = []

    # Seasons to compute (unique + All_Seasons)
    seasons_to_compute = df['Season'].unique().tolist() + ["All_Seasons"]

    # Loop over each EA
    for ea in df[ea_col].unique():
        ea_df_all = df[df[ea_col] == ea].copy()

        for season in seasons_to_compute:
            if season == "All_Seasons":
                ea_df = ea_df_all.copy()
            else:
                ea_df = ea_df_all[ea_df_all['Season'] == season].copy()

            if ea_df.empty:
                continue

            # Track combo counts for filtering
            combo_counts = {}

            # Create exclusive hazard columns for this EA + Season
            for combo in all_combos:
                combo_name = ''.join(combo)
                in_combo = np.logical_and.reduce([ea_df[hazard_cols[h]] for h in combo])
                not_in_combo = np.logical_not(np.logical_or.reduce([ea_df[hazard_cols[h]] 
                                                                   for h in hazard_keys if h not in combo]))
                ea_df[f'{combo_name}_Only_Day'] = in_combo & not_in_combo
                ea_df[f'{combo_name}_Only_Undervolt'] = ea_df[f'{combo_name}_Only_Day'] & ea_df['Undervolt_Day']
                combo_counts[combo_name] = ea_df[f'{combo_name}_Only_Day'].sum()

            # Identify top N for this EA + Season
            top_combos = sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:print_top_n]
            top_combo_names = {name for name, _ in top_combos}

            # Calculate summary for this EA + Season
            ea_row = {
                ea_col: ea,
                'season': season,
                'total_days': len(ea_df),
                'undervolt_dur': undervolt_dur
            }

            for combo in all_combos:
                combo_name = ''.join(combo)
                if filter_top_n_only and combo_name not in top_combo_names:
                    continue

                days_sum = ea_df[f'{combo_name}_Only_Day'].sum()
                undervolt_sum = ea_df[f'{combo_name}_Only_Undervolt'].sum()

                ea_row[f'{combo_name}_days'] = days_sum
                ea_row[f'{combo_name}_n_undervolt'] = undervolt_sum
                ea_row[f'{combo_name}_undervolt_prob'] = p_cond(undervolt_sum, days_sum)

            all_results.append(ea_row)

    return pd.DataFrame(all_results).fillna(0).round(4)

### 1+ hour undervolts 

In [ ]:
# 22
volt_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_60.csv')
volt_hourly_22['time'] = pd.to_datetime(volt_hourly_22['time'])
volt_hourly_22['time'] = volt_hourly_22['time'].dt.tz_convert(None)


# 23
volt_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_60.csv')
volt_hourly_23['time'] = pd.to_datetime(volt_hourly_23['time'])
volt_hourly_23['time'] = volt_hourly_23['time'].dt.tz_convert(None)

pqr_60_min_all = pd.concat([volt_hourly_22, volt_hourly_23], ignore_index=True)

pqr_60_min_all = pqr_60_min_all[ ~(pqr_60_min_all['site_id'].isin(incomplete_site_ids)) ]

#### 95th percentile 

In [ ]:
## Specify percentile 
dur = 60  # duration in minutes
percentile = '95'
voltage_df = pqr_60_min_all   ## 60 minutes / 1 hour 

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPL_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

In [ ]:
daily_agg = create_daily_summary_unv_TPL(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                                       )

global_undervolt_60min_95th_per_EA = calculate_cooccurrence_prob_all_combinations_undervolt_per_EA(daily_agg, undervolt_dur = 60)
global_undervolt_60min_95th_per_EA['Pct'] = f"{percentile}th"

# filter for all seasons 
global_undervolt_60min_95th_per_EA = global_undervolt_60min_95th_per_EA[
                                                global_undervolt_60min_95th_per_EA
                                                    ['season'] == 'All_Seasons'].reset_index(drop=True)


### Columns of Interest 

undervolt_cols_of_interest = ['ea_code9ch', 'undervolt_dur', 'total_days', 
                              'T_days', 'T_n_undervolt', 'T_undervolt_prob', 
                              'P_days', 'P_n_undervolt', 'P_undervolt_prob', 
                              'L_days', 'L_n_undervolt', 'L_undervolt_prob', 
                              'PL_days', 'PL_n_undervolt', 'PL_undervolt_prob', 
                              'Pct']

undervolt_ea_co_occur_95 = global_undervolt_60min_95th_per_EA[undervolt_cols_of_interest]

### 1+ hour co-occurrence df 

In [ ]:
rez_SPATIAL_unv_1hr_list = [undervolt_ea_co_occur_95]
rez_SPATIAL_unv_1hr_TPL = pd.concat(rez_SPATIAL_unv_1hr_list, ignore_index=True)

### 4+ hour undervolts 

In [ ]:
# 22
volt_hourly_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_22_und_240.csv')
volt_hourly_22['time'] = pd.to_datetime(volt_hourly_22['time'])
volt_hourly_22['time'] = volt_hourly_22['time'].dt.tz_convert(None)


# 23
volt_hourly_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/Geospatial_Voltage_Durations/Files/voltage_hourly_23_und_240.csv')
volt_hourly_23['time'] = pd.to_datetime(volt_hourly_23['time'])
volt_hourly_23['time'] = volt_hourly_23['time'].dt.tz_convert(None)


pqr_240_min_all = pd.concat([volt_hourly_22, volt_hourly_23], ignore_index=True)

pqr_240_min_all = pqr_240_min_all[ ~(pqr_240_min_all['site_id'].isin(incomplete_site_ids)) ]

#### 95th percentile 

In [ ]:
## Specify percentile 
dur = 240  # duration in minutes
percentile = '95'
voltage_df = pqr_240_min_all   ## 60 minutes / 1 hour 

results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_df_TPL_n_undervolt_data(row, temp_all, precip_all, hourly_lightning_df, voltage_df)
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data = pd.concat(results, ignore_index=True)

In [ ]:
daily_agg = create_daily_summary_unv_TPL(
                            merged_hourly_data, 
                            temp_thresh=temp_thresh_dict[percentile], 
                            hot_hours_thresh=hot_hrs_thresh_dict[percentile], 
                            precip_thresh=precip_thresh_dict[percentile], 
                            lightning_thresh=lightning_thresh_dict[percentile],  
                                       )

global_undervolt_240min_95th_per_EA = calculate_cooccurrence_prob_all_combinations_undervolt_per_EA(daily_agg, undervolt_dur = dur)
global_undervolt_240min_95th_per_EA['Pct'] = f"{percentile}th"

# filter for all seasons 
global_undervolt_240min_95th_per_EA = global_undervolt_240min_95th_per_EA[
                                                global_undervolt_240min_95th_per_EA
                                                    ['season'] == 'All_Seasons'].reset_index(drop=True)


### Filter columns of interest 
undervolt_ea_co_occur_95 = global_undervolt_240min_95th_per_EA[undervolt_cols_of_interest]

### 4+ hour co-occurrence df 

In [ ]:
rez_SPATIAL_unv_4hr_list = [undervolt_ea_co_occur_95]
rez_SPATIAL_unv_4hr_TPL = pd.concat(rez_SPATIAL_unv_4hr_list, ignore_index=True)

# rez_SPATIAL_unv_4hr_TPL.to_csv('/home/kdonkor_umass_edu/Co-occurrence_Rev2/Files/rez_SPATIAL_unv_4hr_TPL_rev.csv')